In [1]:
from google.colab import drive
# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


# Parameter Opimization

From this stage, in order to replicate the results as closely as possible, I am using the 26 features selected by the paper as the Global BFC (Best Feature Combination). Therefore, parameter tuning is conducted exclusively on this BFC, testing five different C values: ` [0.01, 0.1, 1, 10, 100]` , following the same procedure.


## Loading the Datasets

In [2]:
import os
import re
import shutil
import random
import gc
from ast import literal_eval
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, recall_score, precision_score, f1_score, matthews_corrcoef, roc_auc_score)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Define two different folder paths
project_folder = '/content/drive/MyDrive/CSI5180_Project'
results_folder = '/content/drive/MyDrive/CSI5180_Project/results'

# Read the two result files
summary_df = pd.read_csv(os.path.join(results_folder, 'summary_best_k.csv'))
feature_freq_df = pd.read_csv(os.path.join(results_folder, 'Feature_Frequency_Analysis.csv'))

# Convert the 'feature_rank' column in summary_df from a string to a list
summary_df['feature_rank'] = summary_df['feature_rank'].apply(literal_eval)

# Define dataset storage paths
drive_datasets_folder = '/content/drive/MyDrive/CSI5180_Project/balanced_datasets'
local_datasets_folder = '/content/balanced_datasets'

In [3]:
# Ensure local directory exists
os.makedirs(local_datasets_folder, exist_ok=True)

# Copy all data from Google Drive to local storage
shutil.copytree(drive_datasets_folder, local_datasets_folder, dirs_exist_ok=True)

print("Data copied to /content/balanced_datasets. Training is no longer affected by Google Drive API limitations.")

Data copied to /content/balanced_datasets. Training is no longer affected by Google Drive API limitations.


In [4]:
# Load all datasets
eval_datasets_analysis = {}
for _, row in summary_df.iterrows():
    dataset_path = os.path.join(local_datasets_folder, row['dataset'])
    try:
        df = pd.read_csv(dataset_path)
        eval_datasets_analysis[row['dataset']] = df
    except Exception as e:
        print(f"Failed to load dataset {row['dataset']}: {e}")

# Preprocess datasets
processed_datasets = {}

# Define the 26 selected best features from ESI-Table D of the research paper
global_BFC = [
    "PR", "H3", "RN_CC", "H5", "Alaf", "Argf", "H7", "CAI", "FCA_AS", "FCA_num_triangles",
    "FCA_HS", "Leuf", "RN_BC", "GC1", "T3", "NGSE", "Glyf", "Valf", "Serf", "FCA_ClustCoef",
    "Phef", "C3", "Gluf", "GC2", "Asnf", "FCA_BC"
]

for ds_name, df in eval_datasets_analysis.items():
    try:
        # Only select BFC for the current dataset
        X = df[global_BFC]
        # Extract the target variable 'Class'
        y = df['Class']

        # Map target variable
        mapping = {'E': 1, 'N': 0}
        y = y.map(mapping)
        if y.isnull().any():
            raise ValueError("NaN values found after mapping target variable")

        # Convert data types for memory efficiency
        processed_datasets[ds_name] = {
            'features': X.astype(np.float32),
            'target': y.astype(np.int8)
        }
    except Exception as e:
        print(f"Skipped dataset {ds_name}: {str(e)}")

print(f"Successfully processed {len(processed_datasets)} datasets.")

Successfully processed 1000 datasets.


In [ ]:
# ----------------------------------------
# Function to Process a Single Dataset
# ----------------------------------------
def process_single_dataset_for_c(ds_name, c_value):
    """
    Process a single dataset with a given C value for Logistic Regression.

    Parameters:
    - ds_name: str, the name of the dataset to process
    - c_value: float, the C value for the LogisticRegression model

    Returns:
    - np.mean(fold_auc_scores): float, the average AUC score across all folds
    """

    try:
        # Retrieve the dataset from the global processed_datasets dictionary
        data = processed_datasets[ds_name]

        # Extract features (X) and target labels (y) for the current dataset
        X = data['features']
        y = data['target']

        # Standardize the features for better model performance
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Set up a 10-fold stratified cross-validation
        cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
        fold_auc_scores = []

        for train_idx, test_idx in cv.split(X_scaled, y):
            # Initialize the Logistic Regression model with specified parameters
            model = LogisticRegression(
                C=c_value,
                random_state=RANDOM_STATE,
                max_iter=1000,
                solver='lbfgs',
                penalty='l2'
            )
            # Split data into training and testing sets for the current fold
            X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # Fit the model on the training set
            model.fit(X_train, y_train)
            # Directly predict probabilities for the positive class on the test set
            probabilities = model.predict_proba(X_test)[:, 1]

            # Compute AUC score for the current fold and store it
            fold_auc_scores.append(roc_auc_score(y_test, probabilities))

            # Free memory resources after each fold
            del model, X_train, X_test, y_train, y_test, probabilities

        # Clean up large variables no longer needed
        del X_scaled, X, scaler

        # Return the mean AUC score across all folds
        return np.mean(fold_auc_scores)

    except Exception as err:
        print(f"Error processing dataset {ds_name}: {err}")
        return np.nan

# ----------------------------------------------------
# Evaluation Function Using Parallel Processing
# ----------------------------------------------------
def evaluate_c_value(c_value):
    """
    Parallel evaluation of a given C value across all datasets.

    Modified to also return the individual AUC scores for each dataset.

    Parameters:
    - c_value: float, the C value to evaluate

    Returns:
    - avg_auc: float, the average AUC score across all valid datasets
    - auc_scores: dict, mapping each dataset key to its computed AUC score
    """

    auc_scores = {}
    counter = 0

    # Create a progress bar to track processing progress
    with tqdm(total=len(dataset_keys), desc="Processing:", dynamic_ncols=True) as pbar:
        # Use parallel processing to speed up computation
        with ProcessPoolExecutor(max_workers=2) as executor:
            # Submit tasks for each dataset with the specified C value
            futures = {
                executor.submit(process_single_dataset_for_c, ds_name, c_value): ds_name
                for ds_name in dataset_keys
            }
            # Collect results as tasks complete
            for future in as_completed(futures):
                # Retrieve the dataset name corresponding to the completed future
                ds_name = futures[future]
                # Obtain auROC for a single dataset and store it in the auc_scores dictionary
                auc_scores[ds_name] = future.result()

                counter += 1
                # Perform garbage collection every 5 datasets to free up memory
                if counter % 5 == 0:
                    gc.collect()
                pbar.update(1)

    # Filter out any invalid auROC scores (NaN) from the stored results.
    valid_auc = [score for score in auc_scores.values() if not np.isnan(score)]
    # Compute the overall average auROC score across all valid datasets.
    avg_auc = np.mean(valid_auc) if valid_auc else np.nan

    return avg_auc, auc_scores

In [ ]:
# ---------------------------
# Main Evaluation Loop
# ---------------------------
# List of C values to evaluate during parameter tuning.
c_values = [0.01, 0.1, 1, 10, 100]

# Create a list of dataset keys to be used for tracking progress during evaluation.
dataset_keys = list(processed_datasets.keys())

print("Evaluating 5 different C values for Logistic Regression: [0.01, 0.1, 1, 10, 100]")

c_performance_results = []
c_auc_dict = {}  # Dictionary to store individual auc_scores for each C value

for idx, c_val in enumerate(c_values, start=1):
    print(f"\nEvaluating Logistic Regression with C = {c_val}  [{idx}/{len(c_values)}]")
    avg_auc, auc_scores = evaluate_c_value(c_val)
    c_performance_results.append({
        'C_value': c_val,
        'avg_auc': avg_auc
    })
    c_auc_dict[c_val] = auc_scores  # Save individual AUC scores for this C value
    print(f"Average auROC = {avg_auc:.5f}")

# Determine best C value (Cbest) based on highest average auROC
best_result = max(c_performance_results, key=lambda x: x['avg_auc'])
Cbest = best_result['C_value']
print(f"\nBest C value (C_best) determined: {Cbest}")

# Determine best balanced dataset (BDbest) among the 1000 datasets for the best C value
auc_scores_for_Cbest = c_auc_dict[Cbest]
if auc_scores_for_Cbest:
    best_dataset = max(auc_scores_for_Cbest.items(), key=lambda x: x[1])
    print("\nBest Balanced Dataset (BD_best) with C_best:")
    print(f"{best_dataset[0]} with auROC = {best_dataset[1]:.5f}")
else:
    print("No valid AUC scores found for C_best.")

# Final memory cleanup to free up disk and memory resources after all operations
gc.collect()

Evaluating 5 different C values for Logistic Regression: [0.01, 0.1, 1, 10, 100]

Evaluating Logistic Regression with C = 0.01  [1/5]


Processing:: 100%|██████████| 1000/1000 [02:40<00:00,  6.23it/s]


Average auROC = 0.91639

Evaluating Logistic Regression with C = 0.1  [2/5]


Processing:: 100%|██████████| 1000/1000 [02:14<00:00,  7.46it/s]


Average auROC = 0.93190

Evaluating Logistic Regression with C = 1  [3/5]


...

Processing:: 100%|██████████| 1000/1000 [03:56<00:00,  4.22it/s]


Average auROC = 0.93309

Evaluating Logistic Regression with C = 100  [5/5]


Processing:: 100%|██████████| 1000/1000 [04:00<00:00,  4.17it/s]


Average auROC = 0.93264

Best C value (C_best) determined: 1

Best Balanced Dataset (BD_best) with C_best:
balanced_dataset_239.csv with auROC = 0.95902


0

# Model Training and Testing

In the paper, the authors demonstrate and compare the predicted gene labels obtained under two scenarios:

1. ​**Best Model Predictions**  
   Predictions made by the best model (`BFC_best` , `BD_best`, `C_best`) trained on random datasets. This model is used to predict the phenotype of each reaction–gene pair as Essential (E) or Non-Essential (N).

2. ​**Ensemble Predictions**  
   Predictions made by 1000 trained models (`BFC_best`, `C_best`), each trained on a separate random dataset. Here, each reaction–gene pair is classified as Essential (E) if at least 80% of the models vote for it as essential. (This threshold is user-defined and adjustable.)

The paper uses the primary imbalanced dataset as the test set for both approaches.

## Comparison of Single-Model Probabilities and Ensemble Vote Percentages

- ​**Single-Model Probabilities**  
  Derived from calibrated outputs (e.g., via `predict_proba`), these probabilities accurately reflect the model's confidence and provide a strong basis for ranking samples for auROC computation.

- ​**Ensemble Vote Percentages**  
  Computed as the fraction of models predicting the positive class, these consensus scores can effectively rank samples for auROC if the ensemble distinguishes between classes well.

Both approaches, despite their differences, can be used to compute auROC as long as they maintain a meaningful ordering of samples.

## The Best Model

Calculate the weighted performance metrics as defined in the paper, and compare the results with those reported in the paper.


In [8]:
def train_and_save_best_model(processed_datasets, best_dataset_key, global_BFC, C_best, project_folder, random_state=42):
    """
    Train the final best model using the dataset identified by best_dataset_key,
    using only the features in global_BFC, and save the trained model along with the scaler.

    Parameters:
        processed_datasets (dict): Dictionary containing preprocessed datasets.
        best_dataset_key (str): The key for the best dataset to be used for training.
        global_BFC (list): List of feature names to be used (e.g., the 26 or 35 best features).
        C_best (float): The best penalty parameter C for LogisticRegression.
        project_folder (str): The folder path where the model will be saved.
        random_state (int): Random seed for reproducibility (default=42).

    Returns:
        best_model (dict): A dictionary containing the trained model, scaler, and feature order.
    """

    # Retrieve the dataset from processed_datasets
    data = processed_datasets[best_dataset_key]

    # Extract features (X) and target labels (y)
    X = data['features']
    y = data['target']

    # Standardize the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Initialize the LogisticRegression model with the best C value and L2 regularization
    model = LogisticRegression(
        C=C_best,
        random_state=random_state,
        max_iter=1000,
        solver='lbfgs',
        penalty='l2'
    )

    # Train the model on the entire dataset
    model.fit(X_scaled, y)

    # Prepare the best_model dictionary
    best_model = {
        "model": model,
        "scaler": scaler,
        "features": global_BFC
    }

    # Define the save path and save the model
    save_path = os.path.join(project_folder, "best_model_2nd.pkl")
    joblib.dump(best_model, save_path)
    print(f"Best model saved to {save_path}")

    return best_model

In [9]:
# Use the previously calculated optimal C value and best dataset filename
C_best = 1
best_dataset_key = "balanced_dataset_239.csv"

# Define the folder path in Google Drive where best models are stored
best_model_folder = '/content/drive/MyDrive/CSI5180_Project/best_models'

# Train and save the best model for the 2nd algorithm
best_model_2nd = train_and_save_best_model(processed_datasets, best_dataset_key, global_BFC, C_best, best_model_folder)

Best model saved to /content/drive/MyDrive/CSI5180_Project/best_models/best_model_2nd.pkl


In [10]:
def binary_metrics(y_true, y_pred, y_prob, pos_label=1):
    """
    Compute binary metrics for a given positive label, including:
      - TPR (recall)
      - FPR
      - Precision
      - F1-score
      - MCC
      - auROC

    Parameters:
        y_true: array-like of true labels.
        y_pred: array-like of predicted labels.
        y_prob: array-like of predicted probabilities for each class.
                Assumes y_prob[:,1] is the probability for label '1'.
        pos_label: the label considered as positive (default is 1).

    Returns:
        dict: A dictionary with keys "TPR", "FPR", "precision", "recall", "F1", "MCC", "auROC".
    """
    # Convert true and predicted labels to binary: 1 if equal to pos_label, else 0.
    y_true_binary = np.where(np.array(y_true) == pos_label, 1, 0)
    y_pred_binary = np.where(np.array(y_pred) == pos_label, 1, 0)

    # Calculate TPR (which is the same as recall), precision, F1-score, and MCC using sklearn functions.
    tpr = recall_score(y_true_binary, y_pred_binary)  # TPR is equivalent to recall.
    precision = precision_score(y_true_binary, y_pred_binary)
    f1 = f1_score(y_true_binary, y_pred_binary)
    mcc = matthews_corrcoef(y_true_binary, y_pred_binary)

    # Compute FPR using the confusion matrix.
    tn, fp, fn, tp = confusion_matrix(y_true_binary, y_pred_binary, labels=[0,1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    # Select the probability corresponding to the positive label.
    # If pos_label is 1, use y_prob[:,1]; otherwise, use y_prob[:,0].
    prob_for_pos_label = y_prob[:,1] if pos_label == 1 else y_prob[:,0]

    try:
        auc = roc_auc_score(y_true_binary, prob_for_pos_label)
    except ValueError:
        # If roc_auc_score fails (e.g., if one class is missing), return 0.0.
        auc = 0.0

    return {
        "TPR": tpr,
        "FPR": fpr,
        "precision": precision,
        "recall": tpr,
        "F1": f1,
        "MCC": mcc,
        "auROC": auc
    }

def evaluate_best_model_metrics(best_model_path, test_csv_path, mapping={"E": 1, "N": 0}):
    """
    Loads the saved best model, applies it to the test dataset, and computes
    weighted metrics according to the paper's formula:
    Weighted_Metric_i = (M_iP * PI + M_iN * NI) / (PI + NI)
    where M_iP and M_iN are the metrics computed for positive and negative classes, respectively.

    Parameters:
        best_model_path (str): Path to the saved best model file.
        test_csv_path (str): Path to the test CSV file (e.g., Table_S1.csv).
        mapping (dict): A dictionary to map class labels (default: {"E": 1, "N": 0}).

    Returns:
        dict: A dictionary with weighted metrics: "TPR", "FPR", "precision", "recall", "F1", "MCC", "auROC".
    """

    # -----------------------------
    # Load the Best Model
    # -----------------------------
    best_model = joblib.load(best_model_path)
    second_model = best_model["model"]
    scaler = best_model["scaler"]
    global_BFC = best_model["features"]

    # ----------------------------------------
    # Load the Test Dataset (Table_S1.csv)
    # ----------------------------------------
    table_s1_df = pd.read_csv(test_csv_path)
    X_test = table_s1_df[global_BFC]
    y_test_str = table_s1_df["Class"]
    # Map the class labels ("E"/"N") to numeric values.
    y_test = y_test_str.map(mapping)

    # --------------------------------------------
    # Preprocess Test Data and Make Predictions
    # --------------------------------------------
    # Standardize the test features using the saved scaler.
    X_test_scaled = scaler.transform(X_test)
    # Predict class labels.
    y_pred = second_model.predict(X_test_scaled)
    # Obtain predicted probabilities (required for ROC-AUC).
    y_prob = second_model.predict_proba(X_test_scaled)

    # Compute the number of positive and negative samples in the test set.
    PI = sum(y_test == 1)
    NI = sum(y_test == 0)

    # Compute metrics with positive label defined as 1 and then as 0.
    metrics_pos = binary_metrics(y_test, y_pred, y_prob, pos_label=1)  # When positive label is 1.
    metrics_neg = binary_metrics(y_test, y_pred, y_prob, pos_label=0)  # When positive label is 0.

    # ---------------------------------------------------------------
    # Compute weighted metrics according to the paper's formula:
    # Weighted_Metric_i = (M_iP * PI + M_iN * NI) / (PI + NI)
    # ---------------------------------------------------------------
    weighted_results = {}
    for key in ["TPR", "FPR", "precision", "recall", "F1", "MCC", "auROC"]:
        m_pos = metrics_pos[key]
        m_neg = metrics_neg[key]
        weighted_results[key] = (m_pos * PI + m_neg * NI) / (PI + NI)

    return weighted_results

In [11]:
# File paths
best_model_path = os.path.join(best_model_folder, "best_model_2nd.pkl")  # Trained model
test_csv_path = os.path.join(project_folder, "Table_S1.csv")  # Test data

# Calculate evaluation metrics
results_best_model = evaluate_best_model_metrics(
    best_model_path=best_model_path,
    test_csv_path=test_csv_path
)

# Print weighted metrics
print("Weighted Performance Metrics based on the Best Model")
for metric_name, metric_value in results_best_model.items():
    print(f"{metric_name}: {metric_value:.4f}")

Weighted Performance Metrics based on the Best Model
TPR: 0.8516
FPR: 0.0754
precision: 0.9281
recall: 0.8516
F1: 0.8737
MCC: 0.5648
auROC: 0.9355


# Ensemble of 1000 Trained Model

Recalculate weighted performance metrics using ensemble voting results and compare them with the paper’s findings.


In [12]:
# Define the folder path in Google Drive where models are stored
drive_models_folder = '/content/drive/MyDrive/CSI5180_Project/trained_models_2nd'
os.makedirs(drive_models_folder, exist_ok=True)

# Define a helper function to extract numbers from keys for numerical sortings
def extract_number(s):
    match = re.search(r'\d+', s)
    return int(match.group()) if match else -1

# Retrieve all keys from processed_datasets representing balanced datasets and sort them by numerical order
balanced_dataset_keys = sorted(processed_datasets.keys(), key=extract_number)

# Perform 1000 iterations of model training and save each trained model
for i, ds_key in enumerate(balanced_dataset_keys):
    # Retrieve the dataset from processed_datasets using its key
    data = processed_datasets[ds_key]

    # Extract features (X) and target labels (y)
    X = data['features']
    y = data['target']

    # Standardize the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Initialize the LogisticRegression model with the best C value and L2 regularization
    model = LogisticRegression(
        C=C_best,
        random_state=RANDOM_STATE,
        max_iter=1000,
        solver='lbfgs',
        penalty='l2'
    )

    # Train the model on the entire dataset
    model.fit(X_scaled, y)

    # Prepare the model dictionary (saving model, scaler, and feature order)
    trained_model = {
        "model": model,
        "scaler": scaler,
        "features": global_BFC
    }

    # Define the save path
    save_path = os.path.join(drive_models_folder, f"trained_model_2nd{i}.pkl")
    joblib.dump(trained_model, save_path)

print(f"1000 trained models saved to {drive_models_folder}")

1000 trained models saved to /content/drive/MyDrive/CSI5180_Project/trained_models_2nd


In [13]:
# Define the folder path in Google Drive where models are stored
drive_models_folder = '/content/drive/MyDrive/CSI5180_Project/trained_models_2nd'

# Define the local folder path
local_models_folder = '/content/trained_models_2nd'

# Ensure the local directory exists
os.makedirs(local_models_folder, exist_ok=True)

# Copy the model folder from Google Drive to the local environment
shutil.copytree(drive_models_folder, local_models_folder, dirs_exist_ok=True)

print("Models copied to /content/trained_models_2nd. Testing is no longer affected by Google Drive API limitations.")

Models copied to /content/trained_models_2nd. Testing is no longer affected by Google Drive API limitations.


In [14]:
# User-defined threshold: gene-reaction pair is assigned as essential (E) if
# the vote percentage is at least this threshold, otherwise non-essential (N)
threshold = 0.8

# Retrieve all local model file paths
model_files = [os.path.join(local_models_folder, f) for f in os.listdir(local_models_folder) if f.endswith('.pkl')]

# Load the test dataset (Table_S1.csv)
test_csv_path = os.path.join(project_folder, "Table_S1.csv")
df_test = pd.read_csv(test_csv_path)

# Extract column names for gene-reaction pairs and true class labels
gene_reaction_name = df_test.columns[0]   # Gene-Reaction Pair names
true_class = df_test.columns[-1]          # True class

# Extract the test features using the global_BFC feature list (which the models use)
# global_BFC should be defined previously (a list of column names)
X_test = df_test[global_BFC]

# Map the true class labels to numeric values (assuming {"E": 1, "N": 0})
mapping = {"E": 1, "N": 0}
y_test = df_test[true_class].map(mapping)

# Prepare an array to store predictions from each model
n_samples = X_test.shape[0]
n_models = len(model_files)
predictions = np.zeros((n_samples, n_models), dtype=int)

# For each model, load it and predict the classes for all test samples
for j, model_file in enumerate(model_files):
    model_data = joblib.load(model_file)
    model = model_data["model"]
    scaler = model_data["scaler"]

    # Standardize test features using the saved scaler from the model
    X_test_scaled = scaler.transform(X_test)

    # Predict using the loaded model
    y_pred = model.predict(X_test_scaled)

    # Store the predictions (each column corresponds to one model)
    predictions[:, j] = y_pred

# Compute vote percentage: the fraction of models voting for 'E' (assumed to be represented as 1)
vote_percentage = predictions.mean(axis=1)

# Create an output DataFrame with vote percentage
ensemble_df = pd.DataFrame({
    "Gene Reaction Name": df_test[gene_reaction_name],
    "Vote Percentage of E": vote_percentage,  # fraction of models voting for 'E'
    "True Class": df_test[true_class]
})

# Determine predicted class based solely on vote percentage
# If vote_percentage >= threshold, predict 'E'; otherwise predict 'N'
ensemble_df["Predicted Class"] = np.where(ensemble_df["Vote Percentage of E"] >= threshold, "E", "N")

# Define the output CSV path and save the DataFrame
output_csv_path = os.path.join(results_folder, "model_vote_percentage_2nd.csv")
ensemble_df.to_csv(output_csv_path, index=False)
print(f"Vote Percentage of E for each gene-reaction pair saved to {output_csv_path}")

# Display the first few rows of the result
print("\nEnsemble prediction based on vote percentage:")
display(ensemble_df)

Vote Percentage of E for each gene-reaction pair saved to /content/drive/MyDrive/CSI5180_Project/results/model_vote_percentage_2nd.csv

Ensemble prediction based on vote percentage:


,Gene Reaction Name,Vote Percentage of E,True Class,Predicted Class
0,S4FE4ST_b1683,0.98,N,E
1,2AGPG161tipp_b2835,0.00,N,N
2,GDPtex_b2215,0.00,N,N
3,TTRCYCtpp_b0463,0.00,N,N
4,FECRMtonex_b3006,1.00,N,E
...,...,...,...,...
3499,IPDPS_b0029,1.00,E,E
3500,3OAR160_b1093,1.00,E,E
3501,K2L4Aabctex_b3200,1.00,E,E
3502,PPPGO_b3850,1.00,E,E


In [15]:
def calculate_ensemble_metrics(df):
    """
    Calculate weighted binary metrics based on ensemble vote percentage.

    Parameters:
        df (pd.DataFrame): DataFrame containing:
            - "True Class": the true label ("E" or "N")
            - "Predicted Class": ensemble predicted label ("E" or "N")
            - "Vote Percentage": fraction of models voting for 'E'

    Returns:
        dict: Weighted metrics including "TPR", "FPR", "precision", "recall", "F1", "MCC", "auROC".
    """

    # Map the true and predicted classes to binary values (E -> 1, N -> 0)
    y_true = df["True Class"].map({"E": 1, "N": 0}).values
    y_pred = df["Predicted Class"].map({"E": 1, "N": 0}).values

    # Construct y_prob: use vote percentage as the probability of class E
    prob_E = df["Vote Percentage of E"].values  # probability of E
    y_prob = np.column_stack([1 - prob_E, prob_E])

    # Count positive and negative samples
    PI = (y_true == 1).sum()
    NI = (y_true == 0).sum()

    # Compute metrics with positive label = 1 and then 0
    metrics_pos = binary_metrics(y_true, y_pred, y_prob, pos_label=1)
    metrics_neg = binary_metrics(y_true, y_pred, y_prob, pos_label=0)

    # Compute weighted metrics according to:
    # Weighted_Metric_i = (M_iP * PI + M_iN * NI) / (PI + NI)
    weighted_results = {}
    for key in ["TPR", "FPR", "precision", "recall", "F1", "MCC", "auROC"]:
        weighted_results[key] = (metrics_pos[key] * PI + metrics_neg[key] * NI) / (PI + NI) if (PI + NI) > 0 else 0.0

    return weighted_results

In [16]:
# Print weighted metrics
ensemble_weighted_metrics = calculate_ensemble_metrics(ensemble_df)
print("Weighted Performance Metrics based on an Ensemble of 1000 Trained Models")
for key, value in ensemble_weighted_metrics.items():
    print(f"{key}: {value:.4f}")

Weighted Performance Metrics based on an Ensemble of 1000 Trained Models
TPR: 0.8858
FPR: 0.0985
precision: 0.9322
recall: 0.8858
F1: 0.8997
MCC: 0.6118
auROC: 0.9316


# Performance Metrics Based on Ensemble Model Voting in the Original Paper

In [17]:
# Set the threshold as in the paper
threshold = 0.8

# Concatenate the file path
table_s3_path = os.path.join(project_folder, "Table_S3_E_coli_Simplified.csv")

# Read the CSV file into a DataFrame
df_table_s3 = pd.read_csv(table_s3_path)

# Create a DataFrame for the Essential part, retaining relevant columns and dropping rows with missing Essential_Unique_ID
df_essential = df_table_s3[["Essential_Unique_ID",
                            "Essential_Essential_Predict_Percentage",
                            "Essential_Reference_Classfication"]].dropna(subset=["Essential_Unique_ID"]).copy()
# Rename columns
df_essential.columns = ["Gene Reaction Name", "Vote Percentage of E", "True Class"]

# Create a DataFrame for the NonEssential part, retaining the same column names and dropping rows with missing NonEssential_Unique_ID
df_nonessential = df_table_s3[["NonEssential_Unique_ID",
                               "NonEssential_Essential_Predict_Percentage",
                               "NonEssential_Reference_Classfication"]].dropna(subset=["NonEssential_Unique_ID"]).copy()
# Rename columns
df_nonessential.columns = ["Gene Reaction Name", "Vote Percentage of E", "True Class"]

# Concatenate the two DataFrames vertically
combined_df = pd.concat([df_essential, df_nonessential], axis=0)

# Reset the index (optional)
combined_df.reset_index(drop=True, inplace=True)

# Calculate Predicted Class based on Vote Percentage and threshold
combined_df["Predicted Class"] = np.where(combined_df["Vote Percentage of E"] >= threshold, "E", "N")

# Display the updated DataFrame
print("Combined DataFrame with Predicted Class:")
display(combined_df)

Combined DataFrame with Predicted Class:


,Gene Reaction Name,Vote Percentage of E,True Class,Predicted Class
0,FMNAT_b0025,100.0,E,E
1,RBFK_b0025,100.0,E,E
2,DMPPS_b0029,100.0,E,E
3,IPDPS_b0029,100.0,E,E
4,DHDPRy_b0031,100.0,E,E
...,...,...,...,...
3499,AMPTASECG_b4260,100.0,N,E
3500,TMDPP_b4382,100.0,N,E
3501,PPM_b4383,100.0,N,E
3502,PUNP1_b4384,100.0,N,E


In [18]:
# Print weighted metrics
paper_weighted_metrics = calculate_ensemble_metrics(combined_df)
print("Weighted Performance Metrics Based on the Ensemble from the Original Paper")
for key, value in paper_weighted_metrics.items():
    print(f"{key}: {value:.4f}")

Weighted Performance Metrics Based on the Ensemble from the Original Paper
TPR: 0.7620
FPR: 0.0635
precision: 0.9187
recall: 0.7620
F1: 0.8053
MCC: 0.4609
auROC: 0.9204


In [19]:
# Create a dictionary with each key representing a column
data = {
    "Best Model": results_best_model,
    "Ensemble from this Project": ensemble_weighted_metrics,
    "Ensemble from the Paper": paper_weighted_metrics
}

# Convert the dictionary into a DataFrame
df_metrics = pd.DataFrame(data)

# Optionally, sort the DataFrame by index or re-order rows as needed
df_metrics = df_metrics.reindex(["TPR", "FPR", "precision", "recall", "F1", "MCC", "auROC"])

# Print the resulting DataFrame
display(df_metrics)

# Save the result for writing the report
df_metrics.to_csv("logistics_comparison_metrics.csv", index=True)

,Best Model,Ensemble from this Project,Ensemble from the Paper
TPR,0.851598,0.885845,0.761986
FPR,0.075356,0.098545,0.063549
precision,0.928079,0.932179,0.918683
recall,0.851598,0.885845,0.761986
F1,0.873724,0.899707,0.805251
MCC,0.564792,0.611844,0.460885
auROC,0.935499,0.931642,0.920393
